# Module 2 Exercise: Native Execution Engine On vs Off

Standard Apache Spark runs operators on the JVM, one row at a time, with garbage
collection overhead. Fabric ships a **Native Execution Engine** (NEE) built on
**Gluten** (the Spark plug-in) and **Velox** (the C++ vectorized executor).
When NEE is on, hot-path operators (Scan, Filter, Project, HashAggregate, many
Joins) run as native vectorized code instead of JVM bytecode.

**SQL Server analogue:** this is the same shift SQL Server made with **Batch
Mode** in 2016 (on Columnstore) and 2019 (on Rowstore): row-at-a-time
interpreted execution swapped for vectorized batch processing. You did not
rewrite a query, the engine just got faster.

We will `EXPLAIN` and run the same scan-and-aggregate query twice, once with
NEE on (the Fabric default) and once with it forced off, and compare both the
physical operators and the wall-clock time.


## 1. NEE on, Fabric default

Confirm NEE is enabled, then `EXPLAIN` a scan-and-aggregate over `orders`.


In [ ]:
%%sql
SET spark.native.enabled = true;


In [ ]:
%%sql
EXPLAIN
SELECT customer_id, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id;


Look for **Velox-prefixed** operators in the physical plan: `VeloxScan`,
`VeloxFilter`, `VeloxProject`, `VeloxHashAggregate`. These are the native
vectorized implementations executing in C++ instead of on the JVM.

Now run the query and time it.


In [ ]:
%%sql
SELECT customer_id, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id
ORDER BY total_amount DESC
LIMIT 10;


## 2. NEE off, plain JVM Spark

Disable the native engine for this session, then `EXPLAIN` the identical query.


In [ ]:
%%sql
SET spark.native.enabled = false;


In [ ]:
%%sql
EXPLAIN
SELECT customer_id, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id;


The Velox operators are gone. You should now see the standard JVM operators:
`FileScan parquet`, `HashAggregate`, `Exchange`, `Project`. Same logical plan,
different physical executor.

Run the query again and compare elapsed time to the NEE-on run above.


In [ ]:
%%sql
SELECT customer_id, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id
ORDER BY total_amount DESC
LIMIT 10;


## 3. Re-enable NEE

Always leave NEE on at the end of an exercise so subsequent notebooks see the
Fabric default behaviour.


In [ ]:
%%sql
SET spark.native.enabled = true;


## Key takeaway

- Same query, same plan shape, very different physical executor.
- NEE on: Velox-prefixed operators, vectorized C++ execution, typically 2-5x
  faster on scan-and-aggregate workloads.
- NEE off: standard JVM operators, the open-source Spark baseline.
- When Velox does not yet support an expression, Spark transparently falls back
  to JVM execution for that operator. The plan shows a mix of `Velox*` and
  non-`Velox*` operators. The query still runs, you just lose the speed-up for
  the unsupported part.
- **SQL Server bridge:** this is exactly the Batch Mode story. Same shift,
  different engine, same result: faster execution without rewriting the query.
